In [5]:
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import time
import pennylane as qml
import dimod
import pandas as pd
from itertools import product
from scipy.optimize import minimize

In [2]:
np.random.seed(42)

tickers = ["AAPL", "MSFT", "GOOGL",'JPM' , 'GS']
data = yf.download(tickers,start='2020-01-01', end='2026-01-01')
prices=data["Close"]
log_returns = np.log(prices/prices.shift(1)).dropna()

mean_returns= log_returns.mean() *252
cov_matrix = log_returns.cov() *252
sector_map={0:0,1:0,2:0,3:1,4:1}

n_assets=3
n_bits=2
N=n_assets*n_bits

# cov_matrix.head()
# mean_returns.head()


[*********************100%***********************]  5 of 5 completed


In [6]:
import sys
sys.path.append('/Users/linanachdi/Documents/GitHub/portfolio-optimization-engine')
import portfolio_engine as pe

start= time.time()

mw_weights,mw_ret,mw_vol,mw_sharpe = pe.maximum_sharpe(mean_returns,cov_matrix,sector_map=sector_map, sector_cap=0.6)

mw_time=time.time()-start

print("Constrained Markowitz weights:")
for ticker,w in zip(tickers,mw_weights):
    print(f"{ticker}:{w:.4f}")
print(f"Returns: {mw_ret:.2%} | Vol: {mw_vol:.2%} | Sharpe: {mw_sharpe:.4f}")
print(f"Time taken: {mw_time:.4f}s")

Constrained Markowitz weights:
AAPL:0.0000
MSFT:0.2903
GOOGL:0.3097
JPM:0.0415
GS:0.3585
Returns: 22.61% | Vol: 26.08% | Sharpe: 0.6868
Time taken: 0.0489s


In [ ]:
import qubo_engine as qe

Q= qe.build_qubo_matrix(mean_returns,cov_matrix,lambda_risk=1.0,lambda_budget=0.5,lambda_sector=2.0,sector_map=sector_map,
                        sector_cap=0.6,n_assets=3,n_bits=2)

